**Figure 3: Overhead Scaling with Trajectory Length.** Experiments run the
deterministic coding workload at 54/64/96 calls (2 repeats). (a) per-call latency;
(b) end-to-end wall time; (c) isolation cost relative to bare (black dashed line
marks parity); (d) the read-trace penalty in isolation. Results suggest that
per-call overhead grows only mildly with trajectory length (2.6x to 3.1x over bare)
-- the tax is per-boundary, not super-linear -- and the read-trace multiplier (~2x)
stays the dominant term.


In [ ]:
# ipython -c "%run plot_scaling.ipynb"
# FAST/USENIX line-plot conventions: white panels, boxed top legend, red solid squares = ours.
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

STANDARD_WIDTH = 17.8            # USENIX two-column text width, cm

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['axes.grid'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.edgecolor'] = '0.55'
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.fancybox'] = False

OURS  = dict(color='#c00000', marker='s', linestyle='-',  linewidth=1.0, markersize=3.2)
BASE1 = dict(color='#e78129', marker='x', linestyle=':',  linewidth=0.9, markersize=3.6, markeredgewidth=0.9)
BASE2 = dict(color='#4f9fcf', marker='^', linestyle='-.', linewidth=0.9, markersize=3.2, markerfacecolor='none')
REF   = dict(color='black', linestyle='--', linewidth=0.8)

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'

df = pd.read_csv(RESULTS / 'motivation_scaling.csv')
lengths = sorted(df['length'].unique())

def series(metric, mode):
    return np.asarray([float(df[(df['length'] == l) & (df['mode'] == mode)][metric].iloc[0]) for l in lengths])

bare = series('per_step_mean_ms', 'bare')
nt = series('per_step_mean_ms', 'agenttx_without_read_tracing')
fl = series('per_step_mean_ms', 'agenttx_full')
wall = {m: series('wall_mean_s', m) for m in ['bare', 'agenttx_without_read_tracing', 'agenttx_full']}

fig = plt.figure(dpi=300, figsize=(cm_to_inch(STANDARD_WIDTH), cm_to_inch(7.0)))

ax = plt.subplot(2, 2, 1)
h_bare, = ax.plot(lengths, bare, **REF, marker='x', markersize=3.6, markeredgewidth=0.9, label='bare')
h_nt, = ax.plot(lengths, nt, **BASE2, label='AgentTX no-trace')
h_fl, = ax.plot(lengths, fl, **OURS, label='AgentTX full (ours)')
ax.set_ylabel('Per-call latency (ms)', fontsize=8)
ax.set_xlabel('Trajectory length (# calls)\n(a) Per-call overhead', fontsize=7)

ax = plt.subplot(2, 2, 2)
ax.plot(lengths, wall['bare'], **REF, marker='x', markersize=3.6, markeredgewidth=0.9)
ax.plot(lengths, wall['agenttx_without_read_tracing'], **BASE2)
ax.plot(lengths, wall['agenttx_full'], **OURS)
ax.set_ylabel('Trajectory latency (s)', fontsize=8)
ax.set_xlabel('Trajectory length (# calls)\n(b) End-to-end workload', fontsize=7)

ax = plt.subplot(2, 2, 3)
ax.plot(lengths, nt / bare, **BASE2)
ax.plot(lengths, fl / bare, **OURS)
ax.axhline(1.0, **REF)
ax.set_ylim(0.8, max(fl / bare) * 1.2)
ax.set_ylabel('Overhead vs. bare (x)', fontsize=8)
ax.set_xlabel('Trajectory length (# calls)\n(c) Isolation cost', fontsize=7)

ax = plt.subplot(2, 2, 4)
ax.plot(lengths, fl / nt, **OURS)
ax.set_ylim(1.8, 2.4)
ax.set_ylabel('Full / no-trace (x)', fontsize=8)
ax.set_xlabel('Trajectory length (# calls)\n(d) Read-trace penalty', fontsize=7)

for ax in fig.axes:
    ax.set_xticks(lengths)
    ax.tick_params(axis='both', labelsize=7)

fig.legend(handles=[h_bare, h_nt, h_fl], loc='upper center', bbox_to_anchor=(0.5, 1.03), ncol=3,
           fontsize=7, columnspacing=1.0, handlelength=1.8, handletextpad=0.4, borderpad=0.3)
plt.tight_layout(pad=0.6, h_pad=1.6, w_pad=1.2, rect=[0.0, 0.0, 1.0, 0.93])
plt.savefig(FIGDIR / 'FIG-Motivation-Scaling.pdf', bbox_inches='tight', pad_inches=0.02)
plt.savefig(FIGDIR / 'FIG-Motivation-Scaling.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()

print(f"isolation cost across lengths: {[f'{v:.2f}x' for v in fl / bare]}")
print(f"read-trace penalty:            {[f'{v:.2f}x' for v in fl / nt]}")
